# ZGLMM379 — Centro 4014 — Datalake

**Tabela:** `dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra`
**Transação SAP:** ZGLMM379 · **Domínio:** Trilha de auditoria (change document) das requisicoes de compra
**Filtro:** `cod_centro = '4014'`
**Colunas:** 25 · **Clustering declarado:** `num_requisicao_compra, num_documento`

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** — cada uma consulta a tabela
diretamente. Não há widget, view temporária nem ordem obrigatória.

## O que esta tabela é

Uma **trilha de auditoria** (change document do SAP: CDHDR + CDPOS) das requisições de compra.
Cada linha representa **um campo que mudou** em um item de requisição — não a requisição em si.

Isso significa que a mesma requisição aparece em muitas linhas, uma para cada alteração.

## Alertas de partida _(extraídos dos comentários do schema)_

| # | Alerta |
|---|---|
| 1 | **`dt_criacao_documento` está em `dd/MM/yyyy`** — formato diferente de todas as outras tabelas, que usam `AAAAMMDD` |
| 2 | **`num_requisicao_compra` e `num_item` já vêm sem zeros à esquerda** — o SAP exporta **com** zeros |
| 3 | **`ind_alteracao_valor` tem lógica invertida** em relação à origem (`X`/`1` ⇒ Não) |
| 4 | **`desc_campo_modificado` vem de mapa inline**, não do dicionário SAP — campos novos podem ficar sem descrição |
| 5 | **`desc_item` é o texto atual** (`MAX_BY`), não o histórico — não reflete o valor à época da mudança |
| 6 | **Clustering não inclui o campo modificado** — a chave real provavelmente precisa dele |

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados |
| 2 | Volumetria |
| 3 | Confirmação do filtro |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| 6 | Preenchimento de todas as colunas |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 9 | Perfil numérico |
| 10 | Datas, horas e timestamps |
| 11 | Códigos e zeros à esquerda |
| **12** | **Chaves normalizadas para join** |
| **13** | **Checksum de linha** |
| 14 | Amostra |
| 15 | Distribuição interna |
| 16 | Freshness |
| **18** | **Análises específicas do change document** |
| **19** | **EXTRAÇÃO COMPLETA** |
| 20 | Resumo |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 13.


## 1. Metadados da tabela

In [0]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra;

In [0]:
-- Formato, tamanho e particoes
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra;

In [0]:
-- Ultimas gravacoes
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra LIMIT 20;

## 2. Volumetria e representatividade

Quanto o centro 4014 representa do total.

In [0]:
-- 2. VOLUMETRIA DO CENARIO
SELECT
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra)                       AS linhas_tabela_toda,
  (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra WHERE `cod_centro` = '4014')                  AS linhas_centro_4014,
  ROUND(100.0 * (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra WHERE `cod_centro` = '4014')
              / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra), 4)      AS pct_do_total,
  (SELECT COUNT(DISTINCT `cod_centro`) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra)         AS centros_na_tabela;

## 3. Confirmação do filtro

Confirma que `4014` existe e que não há variação de formato que faça o filtro perder linhas.

**Se retornar mais de uma linha, o filtro `= '4014'` está incompleto.**

In [0]:
-- 3. O FILTRO PEGOU TUDO?
SELECT CAST(`cod_centro` AS STRING)         AS valor_bruto,
       length(CAST(`cod_centro` AS STRING)) AS comprimento,
       COUNT(*)                        AS linhas
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '') = '4014'
   OR trim(CAST(`cod_centro` AS STRING)) = '4014'
GROUP BY CAST(`cod_centro` AS STRING), length(CAST(`cod_centro` AS STRING))
ORDER BY linhas DESC;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional.

**Ponto crítico:** o clustering declara `num_requisicao_compra + num_documento`, mas um
documento de modificação altera **vários campos** de uma vez. A chave real provavelmente
precisa incluir `nm_campo_modificado` — e possivelmente o item.

In [0]:
-- GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'),
g AS (
  SELECT 'num_documento' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_documento` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_requisicao_compra + num_documento' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_requisicao_compra`, `num_documento` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_documento + nm_campo_modificado' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_documento`, `nm_campo_modificado` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_requisicao_compra + num_item_requisicao_compra + num_documento' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_requisicao_compra`, `num_item_requisicao_compra`, `num_documento` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_requisicao_compra + num_item_requisicao_compra + num_documento + nm_campo_modificado' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_requisicao_compra`, `num_item_requisicao_compra`, `num_documento`, `nm_campo_modificado` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014')
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave de clustering `num_requisicao_compra + num_documento`.

**Regra:** linhas idênticas = duplicata real. Linhas distintas = granularidade legítima.

In [0]:
-- 5. CHAVES DUPLICADAS
SELECT `num_requisicao_compra`, `num_documento`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
GROUP BY `num_requisicao_compra`, `num_documento`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [0]:
-- O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH cen AS (
  SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
),
dup AS (
  SELECT `num_requisicao_compra`, `num_documento` FROM cen GROUP BY `num_requisicao_compra`, `num_documento` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM cen c JOIN dup ON c.`num_requisicao_compra` <=> dup.`num_requisicao_compra` AND c.`num_documento` <=> dup.`num_documento`
),
agg AS (
  SELECT `num_requisicao_compra`, `num_documento`,
         COUNT(DISTINCT `cod_empresa`) AS `cod_empresa`,
         COUNT(DISTINCT `cod_centro`) AS `cod_centro`,
         COUNT(DISTINCT `cod_grupo_compradores`) AS `cod_grupo_compradores`,
         COUNT(DISTINCT `num_item_requisicao_compra`) AS `num_item_requisicao_compra`,
         COUNT(DISTINCT `desc_item`) AS `desc_item`,
         COUNT(DISTINCT `cod_classe_objeto`) AS `cod_classe_objeto`,
         COUNT(DISTINCT `nm_usuario_modificador`) AS `nm_usuario_modificador`,
         COUNT(DISTINCT `dt_criacao_documento`) AS `dt_criacao_documento`,
         COUNT(DISTINCT `hr_modificacao`) AS `hr_modificacao`,
         COUNT(DISTINCT `qt_dias_entre_solicitacao_modificacao`) AS `qt_dias_entre_solicitacao_modificacao`,
         COUNT(DISTINCT `cod_transacao`) AS `cod_transacao`,
         COUNT(DISTINCT `nm_tabela`) AS `nm_tabela`,
         COUNT(DISTINCT `desc_tabela`) AS `desc_tabela`,
         COUNT(DISTINCT `cod_chave_tabela`) AS `cod_chave_tabela`,
         COUNT(DISTINCT `nm_campo_modificado`) AS `nm_campo_modificado`,
         COUNT(DISTINCT `desc_campo_modificado`) AS `desc_campo_modificado`,
         COUNT(DISTINCT `desc_valor_novo`) AS `desc_valor_novo`,
         COUNT(DISTINCT `desc_valor_anterior`) AS `desc_valor_anterior`,
         COUNT(DISTINCT `tp_modificacao_item`) AS `tp_modificacao_item`,
         COUNT(DISTINCT `desc_tipo_modificacao_item`) AS `desc_tipo_modificacao_item`,
         COUNT(DISTINCT `ind_alteracao_valor`) AS `ind_alteracao_valor`,
         COUNT(DISTINCT `dh_carga`) AS `dh_carga`,
         COUNT(DISTINCT `dh_atualizacao`) AS `dh_atualizacao`
  FROM d GROUP BY `num_requisicao_compra`, `num_documento`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(23,
    'cod_empresa', MAX(`cod_empresa`),
    'cod_centro', MAX(`cod_centro`),
    'cod_grupo_compradores', MAX(`cod_grupo_compradores`),
    'num_item_requisicao_compra', MAX(`num_item_requisicao_compra`),
    'desc_item', MAX(`desc_item`),
    'cod_classe_objeto', MAX(`cod_classe_objeto`),
    'nm_usuario_modificador', MAX(`nm_usuario_modificador`),
    'dt_criacao_documento', MAX(`dt_criacao_documento`),
    'hr_modificacao', MAX(`hr_modificacao`),
    'qt_dias_entre_solicitacao_modificacao', MAX(`qt_dias_entre_solicitacao_modificacao`),
    'cod_transacao', MAX(`cod_transacao`),
    'nm_tabela', MAX(`nm_tabela`),
    'desc_tabela', MAX(`desc_tabela`),
    'cod_chave_tabela', MAX(`cod_chave_tabela`),
    'nm_campo_modificado', MAX(`nm_campo_modificado`),
    'desc_campo_modificado', MAX(`desc_campo_modificado`),
    'desc_valor_novo', MAX(`desc_valor_novo`),
    'desc_valor_anterior', MAX(`desc_valor_anterior`),
    'tp_modificacao_item', MAX(`tp_modificacao_item`),
    'desc_tipo_modificacao_item', MAX(`desc_tipo_modificacao_item`),
    'ind_alteracao_valor', MAX(`ind_alteracao_valor`),
    'dh_carga', MAX(`dh_carga`),
    'dh_atualizacao', MAX(`dh_atualizacao`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada **neste centro**.

Atenção especial a `desc_valor_anterior` e `desc_valor_novo` — em change document é
**normal** que fiquem vazios conforme o tipo de modificação. Ver seção 18.4.

In [0]:
-- PREENCHIMENTO DE TODAS AS COLUNAS
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'),
perf AS (
  SELECT stack(25,
    'cod_empresa', 'string', COUNT_IF(`cod_empresa` IS NULL), COUNT_IF(`cod_empresa` IS NOT NULL AND lower(trim(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_empresa`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_grupo_compradores', 'string', COUNT_IF(`cod_grupo_compradores` IS NULL), COUNT_IF(`cod_grupo_compradores` IS NOT NULL AND lower(trim(`cod_grupo_compradores`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_grupo_compradores`) RLIKE '^0+([.,]0+)?$'),
    'num_requisicao_compra', 'string', COUNT_IF(`num_requisicao_compra` IS NULL), COUNT_IF(`num_requisicao_compra` IS NOT NULL AND lower(trim(`num_requisicao_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_requisicao_compra`) RLIKE '^0+([.,]0+)?$'),
    'num_item_requisicao_compra', 'string', COUNT_IF(`num_item_requisicao_compra` IS NULL), COUNT_IF(`num_item_requisicao_compra` IS NOT NULL AND lower(trim(`num_item_requisicao_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_requisicao_compra`) RLIKE '^0+([.,]0+)?$'),
    'desc_item', 'string', COUNT_IF(`desc_item` IS NULL), COUNT_IF(`desc_item` IS NOT NULL AND lower(trim(`desc_item`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_item`) RLIKE '^0+([.,]0+)?$'),
    'cod_classe_objeto', 'string', COUNT_IF(`cod_classe_objeto` IS NULL), COUNT_IF(`cod_classe_objeto` IS NOT NULL AND lower(trim(`cod_classe_objeto`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_classe_objeto`) RLIKE '^0+([.,]0+)?$'),
    'num_documento', 'string', COUNT_IF(`num_documento` IS NULL), COUNT_IF(`num_documento` IS NOT NULL AND lower(trim(`num_documento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_documento`) RLIKE '^0+([.,]0+)?$'),
    'nm_usuario_modificador', 'string', COUNT_IF(`nm_usuario_modificador` IS NULL), COUNT_IF(`nm_usuario_modificador` IS NOT NULL AND lower(trim(`nm_usuario_modificador`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_usuario_modificador`) RLIKE '^0+([.,]0+)?$'),
    'dt_criacao_documento', 'string', COUNT_IF(`dt_criacao_documento` IS NULL), COUNT_IF(`dt_criacao_documento` IS NOT NULL AND lower(trim(`dt_criacao_documento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_criacao_documento`) RLIKE '^0+([.,]0+)?$'),
    'hr_modificacao', 'string', COUNT_IF(`hr_modificacao` IS NULL), COUNT_IF(`hr_modificacao` IS NOT NULL AND lower(trim(`hr_modificacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`hr_modificacao`) RLIKE '^0+([.,]0+)?$'),
    'qt_dias_entre_solicitacao_modificacao', 'int', COUNT_IF(`qt_dias_entre_solicitacao_modificacao` IS NULL), 0L, COUNT_IF(`qt_dias_entre_solicitacao_modificacao` = 0),
    'cod_transacao', 'string', COUNT_IF(`cod_transacao` IS NULL), COUNT_IF(`cod_transacao` IS NOT NULL AND lower(trim(`cod_transacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_transacao`) RLIKE '^0+([.,]0+)?$'),
    'nm_tabela', 'string', COUNT_IF(`nm_tabela` IS NULL), COUNT_IF(`nm_tabela` IS NOT NULL AND lower(trim(`nm_tabela`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_tabela`) RLIKE '^0+([.,]0+)?$'),
    'desc_tabela', 'string', COUNT_IF(`desc_tabela` IS NULL), COUNT_IF(`desc_tabela` IS NOT NULL AND lower(trim(`desc_tabela`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_tabela`) RLIKE '^0+([.,]0+)?$'),
    'cod_chave_tabela', 'string', COUNT_IF(`cod_chave_tabela` IS NULL), COUNT_IF(`cod_chave_tabela` IS NOT NULL AND lower(trim(`cod_chave_tabela`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_chave_tabela`) RLIKE '^0+([.,]0+)?$'),
    'nm_campo_modificado', 'string', COUNT_IF(`nm_campo_modificado` IS NULL), COUNT_IF(`nm_campo_modificado` IS NOT NULL AND lower(trim(`nm_campo_modificado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_campo_modificado`) RLIKE '^0+([.,]0+)?$'),
    'desc_campo_modificado', 'string', COUNT_IF(`desc_campo_modificado` IS NULL), COUNT_IF(`desc_campo_modificado` IS NOT NULL AND lower(trim(`desc_campo_modificado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_campo_modificado`) RLIKE '^0+([.,]0+)?$'),
    'desc_valor_novo', 'string', COUNT_IF(`desc_valor_novo` IS NULL), COUNT_IF(`desc_valor_novo` IS NOT NULL AND lower(trim(`desc_valor_novo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_valor_novo`) RLIKE '^0+([.,]0+)?$'),
    'desc_valor_anterior', 'string', COUNT_IF(`desc_valor_anterior` IS NULL), COUNT_IF(`desc_valor_anterior` IS NOT NULL AND lower(trim(`desc_valor_anterior`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_valor_anterior`) RLIKE '^0+([.,]0+)?$'),
    'tp_modificacao_item', 'string', COUNT_IF(`tp_modificacao_item` IS NULL), COUNT_IF(`tp_modificacao_item` IS NOT NULL AND lower(trim(`tp_modificacao_item`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_modificacao_item`) RLIKE '^0+([.,]0+)?$'),
    'desc_tipo_modificacao_item', 'string', COUNT_IF(`desc_tipo_modificacao_item` IS NULL), COUNT_IF(`desc_tipo_modificacao_item` IS NOT NULL AND lower(trim(`desc_tipo_modificacao_item`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_tipo_modificacao_item`) RLIKE '^0+([.,]0+)?$'),
    'ind_alteracao_valor', 'string', COUNT_IF(`ind_alteracao_valor` IS NULL), COUNT_IF(`ind_alteracao_valor` IS NOT NULL AND lower(trim(`ind_alteracao_valor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_alteracao_valor`) RLIKE '^0+([.,]0+)?$'),
    'dh_carga', 'timestamp', COUNT_IF(`dh_carga` IS NULL), 0L, 0L,
    'dh_atualizacao', 'timestamp', COUNT_IF(`dh_atualizacao` IS NULL), 0L, 0L
  ) AS (coluna, tipo, nulos, vazios, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros,
       t.total - p.nulos - p.vazios - p.zeros                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                       THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                              AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade

In [0]:
-- CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'),
card AS (
  SELECT stack(25,
    'cod_empresa', 'string', approx_count_distinct(`cod_empresa`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'cod_grupo_compradores', 'string', approx_count_distinct(`cod_grupo_compradores`),
    'num_requisicao_compra', 'string', approx_count_distinct(`num_requisicao_compra`),
    'num_item_requisicao_compra', 'string', approx_count_distinct(`num_item_requisicao_compra`),
    'desc_item', 'string', approx_count_distinct(`desc_item`),
    'cod_classe_objeto', 'string', approx_count_distinct(`cod_classe_objeto`),
    'num_documento', 'string', approx_count_distinct(`num_documento`),
    'nm_usuario_modificador', 'string', approx_count_distinct(`nm_usuario_modificador`),
    'dt_criacao_documento', 'string', approx_count_distinct(`dt_criacao_documento`),
    'hr_modificacao', 'string', approx_count_distinct(`hr_modificacao`),
    'qt_dias_entre_solicitacao_modificacao', 'int', approx_count_distinct(`qt_dias_entre_solicitacao_modificacao`),
    'cod_transacao', 'string', approx_count_distinct(`cod_transacao`),
    'nm_tabela', 'string', approx_count_distinct(`nm_tabela`),
    'desc_tabela', 'string', approx_count_distinct(`desc_tabela`),
    'cod_chave_tabela', 'string', approx_count_distinct(`cod_chave_tabela`),
    'nm_campo_modificado', 'string', approx_count_distinct(`nm_campo_modificado`),
    'desc_campo_modificado', 'string', approx_count_distinct(`desc_campo_modificado`),
    'desc_valor_novo', 'string', approx_count_distinct(`desc_valor_novo`),
    'desc_valor_anterior', 'string', approx_count_distinct(`desc_valor_anterior`),
    'tp_modificacao_item', 'string', approx_count_distinct(`tp_modificacao_item`),
    'desc_tipo_modificacao_item', 'string', approx_count_distinct(`desc_tipo_modificacao_item`),
    'ind_alteracao_valor', 'string', approx_count_distinct(`ind_alteracao_valor`),
    'dh_carga', 'timestamp', approx_count_distinct(`dh_carga`),
    'dh_atualizacao', 'timestamp', approx_count_distinct(`dh_atualizacao`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma.

In [0]:
-- DOMINIO DAS CATEGORICAS
(SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014' GROUP BY `cod_empresa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_grupo_compradores' AS coluna, CAST(`cod_grupo_compradores` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014' GROUP BY `cod_grupo_compradores` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_classe_objeto' AS coluna, CAST(`cod_classe_objeto` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014' GROUP BY `cod_classe_objeto` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_transacao' AS coluna, CAST(`cod_transacao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014' GROUP BY `cod_transacao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'nm_tabela' AS coluna, CAST(`nm_tabela` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014' GROUP BY `nm_tabela` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'desc_tabela' AS coluna, CAST(`desc_tabela` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014' GROUP BY `desc_tabela` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'nm_campo_modificado' AS coluna, CAST(`nm_campo_modificado` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014' GROUP BY `nm_campo_modificado` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'desc_campo_modificado' AS coluna, CAST(`desc_campo_modificado` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014' GROUP BY `desc_campo_modificado` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_modificacao_item' AS coluna, CAST(`tp_modificacao_item` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014' GROUP BY `tp_modificacao_item` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'desc_tipo_modificacao_item' AS coluna, CAST(`desc_tipo_modificacao_item` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014' GROUP BY `desc_tipo_modificacao_item` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_alteracao_valor' AS coluna, CAST(`ind_alteracao_valor` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014' GROUP BY `ind_alteracao_valor` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'nm_usuario_modificador' AS coluna, CAST(`nm_usuario_modificador` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014' GROUP BY `nm_usuario_modificador` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Perfil dos campos numéricos

A tabela tem apenas um campo numérico: `qt_dias_entre_solicitacao_modificacao` (`int`),
que é derivado — ver seção 18.8.

In [0]:
-- PERFIL NUMERICO
SELECT * FROM (
  SELECT stack(1,
    'qt_dias_entre_solicitacao_modificacao', 'int', COUNT(`qt_dias_entre_solicitacao_modificacao`), CAST(MIN(`qt_dias_entre_solicitacao_modificacao`) AS DOUBLE), CAST(MAX(`qt_dias_entre_solicitacao_modificacao`) AS DOUBLE), CAST(AVG(`qt_dias_entre_solicitacao_modificacao`) AS DOUBLE), CAST(percentile_approx(`qt_dias_entre_solicitacao_modificacao`, 0.5) AS DOUBLE), CAST(percentile_approx(`qt_dias_entre_solicitacao_modificacao`, 0.95) AS DOUBLE), COUNT_IF(`qt_dias_entre_solicitacao_modificacao` < 0), COUNT_IF(`qt_dias_entre_solicitacao_modificacao` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, p95, negativos, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
)
ORDER BY coluna;

## 10. Datas, horas e timestamps

**Atenção:** esta tabela usa `dd/MM/yyyy`, formato diferente das demais.
Análise detalhada na seção 18.2.

In [0]:
-- 10. DATA E HORA (formato dd/MM/yyyy e HH:mm:ss)
SELECT 'dt_criacao_documento' AS coluna,
       COUNT_IF(dt_criacao_documento IS NULL OR trim(dt_criacao_documento) = '') AS vazios,
       COUNT_IF(trim(dt_criacao_documento) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$') AS formato_esperado,
       COUNT(DISTINCT dt_criacao_documento) AS valores_distintos,
       MIN(dt_criacao_documento) AS menor_texto,
       MAX(dt_criacao_documento) AS maior_texto
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
UNION ALL
SELECT 'hr_modificacao',
       COUNT_IF(hr_modificacao IS NULL OR trim(hr_modificacao) = ''),
       COUNT_IF(trim(hr_modificacao) RLIKE '^[0-9]{2}:[0-9]{2}:[0-9]{2}$'),
       COUNT(DISTINCT hr_modificacao),
       MIN(hr_modificacao), MAX(hr_modificacao)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014';

In [0]:
-- 10.1 TIMESTAMPS DE CARGA
SELECT MIN(dh_carga)                  AS carga_min,
       MAX(dh_carga)                  AS carga_max,
       COUNT(DISTINCT dh_carga)       AS cargas_distintas,
       MIN(dh_atualizacao)            AS atualizacao_min,
       MAX(dh_atualizacao)            AS atualizacao_max,
       COUNT_IF(dh_carga <> dh_atualizacao) AS carga_diferente_de_atualizacao,
       COUNT_IF(dh_carga IS NULL)     AS carga_nula
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014';

## 11. Códigos — zeros à esquerda e formato

**Atenção:** o comentário declara que `num_requisicao_compra` e `num_item_requisicao_compra`
já vêm **sem zeros à esquerda**. O SAP exporta **com** zeros — normalizar o lado SAP,
não o Datalake.

In [0]:
-- CODIGOS: ZEROS A ESQUERDA E FORMATO
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN com_zeros_esq > 0 THEN 'tem zeros a esquerda' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(6,
    'num_requisicao_compra', 'string', COUNT_IF(CAST(`num_requisicao_compra` AS STRING) IS NULL OR trim(CAST(`num_requisicao_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_requisicao_compra` AS STRING)))), MAX(length(trim(CAST(`num_requisicao_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_requisicao_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_requisicao_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_requisicao_compra` AS STRING)), '^0+', '')),
    'num_item_requisicao_compra', 'string', COUNT_IF(CAST(`num_item_requisicao_compra` AS STRING) IS NULL OR trim(CAST(`num_item_requisicao_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_requisicao_compra` AS STRING)))), MAX(length(trim(CAST(`num_item_requisicao_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_item_requisicao_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_item_requisicao_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_requisicao_compra` AS STRING)), '^0+', '')),
    'num_documento', 'string', COUNT_IF(CAST(`num_documento` AS STRING) IS NULL OR trim(CAST(`num_documento` AS STRING)) = ''), MIN(length(trim(CAST(`num_documento` AS STRING)))), MAX(length(trim(CAST(`num_documento` AS STRING)))), COUNT_IF(trim(CAST(`num_documento` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_documento` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_documento` AS STRING)), '^0+', '')),
    'cod_chave_tabela', 'string', COUNT_IF(CAST(`cod_chave_tabela` AS STRING) IS NULL OR trim(CAST(`cod_chave_tabela` AS STRING)) = ''), MIN(length(trim(CAST(`cod_chave_tabela` AS STRING)))), MAX(length(trim(CAST(`cod_chave_tabela` AS STRING)))), COUNT_IF(trim(CAST(`cod_chave_tabela` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_chave_tabela` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_chave_tabela` AS STRING)), '^0+', '')),
    'cod_centro', 'string', COUNT_IF(CAST(`cod_centro` AS STRING) IS NULL OR trim(CAST(`cod_centro` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro` AS STRING)))), MAX(length(trim(CAST(`cod_centro` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '')),
    'cod_empresa', 'string', COUNT_IF(CAST(`cod_empresa` AS STRING) IS NULL OR trim(CAST(`cod_empresa` AS STRING)) = ''), MIN(length(trim(CAST(`cod_empresa` AS STRING)))), MAX(length(trim(CAST(`cod_empresa` AS STRING)))), COUNT_IF(trim(CAST(`cod_empresa` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_empresa` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_empresa` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
)
ORDER BY coluna;

## 12. Chaves normalizadas para join com o SAP

Chave completa já sem zeros à esquerda, pronta para cruzar com o extrato do SAP.

In [0]:
-- 12. CHAVES NORMALIZADAS
SELECT DISTINCT
       regexp_replace(trim(CAST(`num_requisicao_compra` AS STRING)), '^0+', '') AS `num_requisicao_compra_norm`,
       regexp_replace(trim(CAST(`num_item_requisicao_compra` AS STRING)), '^0+', '') AS `num_item_requisicao_compra_norm`,
       regexp_replace(trim(CAST(`num_documento` AS STRING)), '^0+', '') AS `num_documento_norm`
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
ORDER BY 1, 2, 3;

## 13. Checksum de linha

Se `linhas` > `linhas_unicas`, existem registros 100% idênticos — duplicata real.

In [0]:
-- CHECKSUM DE LINHA
SELECT COUNT(*)                                            AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`num_requisicao_compra` AS STRING), ''), COALESCE(CAST(`num_item_requisicao_compra` AS STRING), ''), COALESCE(CAST(`desc_item` AS STRING), ''), COALESCE(CAST(`cod_classe_objeto` AS STRING), ''), COALESCE(CAST(`num_documento` AS STRING), ''), COALESCE(CAST(`nm_usuario_modificador` AS STRING), ''), COALESCE(CAST(`dt_criacao_documento` AS STRING), ''), COALESCE(CAST(`hr_modificacao` AS STRING), ''), COALESCE(CAST(`qt_dias_entre_solicitacao_modificacao` AS STRING), ''), COALESCE(CAST(`cod_transacao` AS STRING), ''), COALESCE(CAST(`nm_tabela` AS STRING), ''), COALESCE(CAST(`desc_tabela` AS STRING), ''), COALESCE(CAST(`cod_chave_tabela` AS STRING), ''), COALESCE(CAST(`nm_campo_modificado` AS STRING), ''), COALESCE(CAST(`desc_campo_modificado` AS STRING), ''), COALESCE(CAST(`desc_valor_novo` AS STRING), ''), COALESCE(CAST(`desc_valor_anterior` AS STRING), ''), COALESCE(CAST(`tp_modificacao_item` AS STRING), ''), COALESCE(CAST(`desc_tipo_modificacao_item` AS STRING), ''), COALESCE(CAST(`ind_alteracao_valor` AS STRING), ''), COALESCE(CAST(`dh_carga` AS STRING), ''), COALESCE(CAST(`dh_atualizacao` AS STRING), ''))))            AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`num_requisicao_compra` AS STRING), ''), COALESCE(CAST(`num_item_requisicao_compra` AS STRING), ''), COALESCE(CAST(`desc_item` AS STRING), ''), COALESCE(CAST(`cod_classe_objeto` AS STRING), ''), COALESCE(CAST(`num_documento` AS STRING), ''), COALESCE(CAST(`nm_usuario_modificador` AS STRING), ''), COALESCE(CAST(`dt_criacao_documento` AS STRING), ''), COALESCE(CAST(`hr_modificacao` AS STRING), ''), COALESCE(CAST(`qt_dias_entre_solicitacao_modificacao` AS STRING), ''), COALESCE(CAST(`cod_transacao` AS STRING), ''), COALESCE(CAST(`nm_tabela` AS STRING), ''), COALESCE(CAST(`desc_tabela` AS STRING), ''), COALESCE(CAST(`cod_chave_tabela` AS STRING), ''), COALESCE(CAST(`nm_campo_modificado` AS STRING), ''), COALESCE(CAST(`desc_campo_modificado` AS STRING), ''), COALESCE(CAST(`desc_valor_novo` AS STRING), ''), COALESCE(CAST(`desc_valor_anterior` AS STRING), ''), COALESCE(CAST(`tp_modificacao_item` AS STRING), ''), COALESCE(CAST(`desc_tipo_modificacao_item` AS STRING), ''), COALESCE(CAST(`ind_alteracao_valor` AS STRING), ''), COALESCE(CAST(`dh_carga` AS STRING), ''), COALESCE(CAST(`dh_atualizacao` AS STRING), '')))) AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`num_requisicao_compra` AS STRING), ''), COALESCE(CAST(`num_item_requisicao_compra` AS STRING), ''), COALESCE(CAST(`desc_item` AS STRING), ''), COALESCE(CAST(`cod_classe_objeto` AS STRING), ''), COALESCE(CAST(`num_documento` AS STRING), ''), COALESCE(CAST(`nm_usuario_modificador` AS STRING), ''), COALESCE(CAST(`dt_criacao_documento` AS STRING), ''), COALESCE(CAST(`hr_modificacao` AS STRING), ''), COALESCE(CAST(`qt_dias_entre_solicitacao_modificacao` AS STRING), ''), COALESCE(CAST(`cod_transacao` AS STRING), ''), COALESCE(CAST(`nm_tabela` AS STRING), ''), COALESCE(CAST(`desc_tabela` AS STRING), ''), COALESCE(CAST(`cod_chave_tabela` AS STRING), ''), COALESCE(CAST(`nm_campo_modificado` AS STRING), ''), COALESCE(CAST(`desc_campo_modificado` AS STRING), ''), COALESCE(CAST(`desc_valor_novo` AS STRING), ''), COALESCE(CAST(`desc_valor_anterior` AS STRING), ''), COALESCE(CAST(`tp_modificacao_item` AS STRING), ''), COALESCE(CAST(`desc_tipo_modificacao_item` AS STRING), ''), COALESCE(CAST(`ind_alteracao_valor` AS STRING), ''), COALESCE(CAST(`dh_carga` AS STRING), ''), COALESCE(CAST(`dh_atualizacao` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014';

## 14. Amostra de linhas completas

In [0]:
-- 14. AMOSTRA
SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
ORDER BY `num_requisicao_compra`, `num_item_requisicao_compra`, `num_documento`
LIMIT 20;

## 15. Distribuição interna

In [0]:
-- 15. DISTRIBUICAO POR cod_grupo_compradores
SELECT COALESCE(NULLIF(trim(CAST(`cod_grupo_compradores` AS STRING)), ''), '(vazio)') AS `cod_grupo_compradores`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_grupo_compradores` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 15. DISTRIBUICAO POR cod_empresa
SELECT COALESCE(NULLIF(trim(CAST(`cod_empresa` AS STRING)), ''), '(vazio)') AS `cod_empresa`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_empresa` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 15. DISTRIBUICAO POR nm_campo_modificado
SELECT COALESCE(NULLIF(trim(CAST(`nm_campo_modificado` AS STRING)), ''), '(vazio)') AS `nm_campo_modificado`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(CAST(`nm_campo_modificado` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 15. DISTRIBUICAO POR tp_modificacao_item
SELECT COALESCE(NULLIF(trim(CAST(`tp_modificacao_item` AS STRING)), ''), '(vazio)') AS `tp_modificacao_item`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(CAST(`tp_modificacao_item` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 15. DISTRIBUICAO POR cod_transacao
SELECT COALESCE(NULLIF(trim(CAST(`cod_transacao` AS STRING)), ''), '(vazio)') AS `cod_transacao`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_transacao` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

## 16. Freshness

Esta tabela **não tem** `dateingest`. Use `dh_carga` e `dh_atualizacao`.

In [0]:
-- 16. FRESHNESS POR DIA DE CARGA
SELECT to_date(dh_carga)                       AS dia_carga,
       COUNT(*)                                AS linhas,
       COUNT(DISTINCT num_requisicao_compra)   AS requisicoes
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
GROUP BY to_date(dh_carga)
ORDER BY dia_carga DESC
LIMIT 30;

## 18. Análises específicas — ZGLMM379 (change document)

Esta tabela é uma **trilha de auditoria**: cada linha é um campo que mudou em um item de
requisição. O schema declara várias transformações no pipeline — as células abaixo verificam
se cada uma foi aplicada corretamente.

### 18.1 Projeção de `cod_chave_tabela`

O comentário declara o layout `MANDT(3) + BANFN(10) + BNFPO(5)` e diz que
`num_requisicao_compra` e `num_item_requisicao_compra` foram **projetados** das posições
4-13 e 14-18 dessa chave.

Esta célula confirma se a projeção está correta em todas as linhas.

In [0]:
-- 18.1 VALIDACAO DA PROJECAO DE cod_chave_tabela
SELECT COUNT(*)                                                    AS linhas,
       COUNT_IF(length(trim(cod_chave_tabela)) = 18)               AS chave_com_18_posicoes,
       COUNT_IF(length(trim(cod_chave_tabela)) <> 18)              AS chave_com_tamanho_diferente,
       COUNT_IF(regexp_replace(substring(trim(cod_chave_tabela), 4, 10), '^0+', '')
                = trim(num_requisicao_compra))                     AS requisicao_projetada_ok,
       COUNT_IF(regexp_replace(substring(trim(cod_chave_tabela), 4, 10), '^0+', '')
                <> trim(num_requisicao_compra))                    AS requisicao_projetada_divergente,
       COUNT_IF(regexp_replace(substring(trim(cod_chave_tabela), 14, 5), '^0+', '')
                = trim(num_item_requisicao_compra))                AS item_projetado_ok,
       COUNT_IF(regexp_replace(substring(trim(cod_chave_tabela), 14, 5), '^0+', '')
                <> trim(num_item_requisicao_compra))               AS item_projetado_divergente
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014';

In [0]:
-- 18.1b EXEMPLOS DE PROJECAO DIVERGENTE
SELECT cod_chave_tabela,
       length(trim(cod_chave_tabela))                                        AS tamanho,
       substring(trim(cod_chave_tabela), 1, 3)                               AS mandt,
       substring(trim(cod_chave_tabela), 4, 10)                              AS banfn_bruto,
       regexp_replace(substring(trim(cod_chave_tabela), 4, 10), '^0+', '')   AS banfn_sem_zeros,
       num_requisicao_compra,
       substring(trim(cod_chave_tabela), 14, 5)                              AS bnfpo_bruto,
       regexp_replace(substring(trim(cod_chave_tabela), 14, 5), '^0+', '')   AS bnfpo_sem_zeros,
       num_item_requisicao_compra
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
  AND (regexp_replace(substring(trim(cod_chave_tabela), 4, 10), '^0+', '') <> trim(num_requisicao_compra)
    OR regexp_replace(substring(trim(cod_chave_tabela), 14, 5), '^0+', '') <> trim(num_item_requisicao_compra))
LIMIT 20;

### 18.2 Formato da data — ATENÇÃO ESPECIAL

O comentário declara que `dt_criacao_documento` foi **convertida de `yyyyMMdd` para `dd/MM/yyyy`**.

Isso é **diferente de todas as outras tabelas** do Datalake, que mantêm `AAAAMMDD`.
Se o extrato do SAP vier em `yyyyMMdd` ou `dd.MM.yyyy`, a comparação direta acusa
100% de divergência sem que haja erro real.

In [0]:
-- 18.2 FORMATO DA DATA E DA HORA
SELECT COUNT(*)                                                             AS linhas,
       COUNT_IF(trim(dt_criacao_documento) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$') AS fmt_dd_MM_yyyy,
       COUNT_IF(trim(dt_criacao_documento) RLIKE '^[0-9]{8}$')            AS fmt_yyyyMMdd,
       COUNT_IF(trim(dt_criacao_documento) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}') AS fmt_ISO,
       COUNT_IF(dt_criacao_documento IS NULL OR trim(dt_criacao_documento) = '') AS vazios,
       COUNT_IF(trim(hr_modificacao) RLIKE '^[0-9]{2}:[0-9]{2}:[0-9]{2}$') AS hora_HH_mm_ss,
       COUNT_IF(trim(hr_modificacao) RLIKE '^[0-9]{6}$')                  AS hora_HHmmss
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014';

In [0]:
-- 18.2b JANELA TEMPORAL (converte dd/MM/yyyy para date)
SELECT MIN(to_date(dt_criacao_documento, 'dd/MM/yyyy'))       AS data_mais_antiga,
       MAX(to_date(dt_criacao_documento, 'dd/MM/yyyy'))       AS data_mais_recente,
       COUNT(DISTINCT dt_criacao_documento)                    AS datas_distintas,
       COUNT_IF(to_date(dt_criacao_documento, 'dd/MM/yyyy') IS NULL
            AND trim(dt_criacao_documento) <> '')              AS nao_convertidas,
       COUNT_IF(to_date(dt_criacao_documento, 'dd/MM/yyyy') > current_date()) AS datas_futuras
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014';

### 18.3 Domínio do tipo de modificação

O comentário declara o domínio `CDCHNGIND`: **U** (modificação), **I**/**J** (inserção),
**E**/**D** (eliminação). Valores fora desse conjunto indicam dado inesperado.

A célula também verifica se o de-para para `desc_tipo_modificacao_item` é consistente.

In [0]:
-- 18.3 DOMINIO DO TIPO DE MODIFICACAO
SELECT tp_modificacao_item,
       desc_tipo_modificacao_item,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'), 2) AS pct,
       CASE WHEN tp_modificacao_item IN ('U','I','J','E','D') THEN 'dominio esperado'
            ELSE 'FORA DO DOMINIO CDCHNGIND' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
GROUP BY tp_modificacao_item, desc_tipo_modificacao_item
ORDER BY linhas DESC;

In [0]:
-- 18.3b CONSISTENCIA DO DE-PARA tp_modificacao_item -> desc
SELECT tp_modificacao_item,
       COUNT(DISTINCT desc_tipo_modificacao_item) AS qtd_descricoes,
       CONCAT_WS(' | ', SORT_ARRAY(COLLECT_SET(desc_tipo_modificacao_item))) AS descricoes,
       CASE WHEN COUNT(DISTINCT desc_tipo_modificacao_item) > 1
            THEN 'ERRO - codigo com mais de uma descricao' ELSE 'ok' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
GROUP BY tp_modificacao_item
ORDER BY qtd_descricoes DESC;

### 18.4 Coerência entre tipo de modificação e valores

Regra esperada do change document:

| Tipo | `desc_valor_anterior` | `desc_valor_novo` |
|---|---|---|
| `I` / `J` (inserção) | vazio | preenchido |
| `E` / `D` (eliminação) | preenchido | vazio |
| `U` (modificação) | preenchido | preenchido, e **diferente** do anterior |

Violações indicam problema de carga ou comportamento não documentado.

In [0]:
-- 18.4 TIPO DE MODIFICACAO x VALORES
SELECT 'I/J (insercao) COM valor anterior preenchido' AS situacao,
       COUNT_IF(tp_modificacao_item IN ('I','J')
            AND desc_valor_anterior IS NOT NULL AND trim(desc_valor_anterior) <> '') AS linhas
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
UNION ALL
SELECT 'E/D (eliminacao) COM valor novo preenchido',
       COUNT_IF(tp_modificacao_item IN ('E','D')
            AND desc_valor_novo IS NOT NULL AND trim(desc_valor_novo) <> '')
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
UNION ALL
SELECT 'U (modificacao) SEM valor novo',
       COUNT_IF(tp_modificacao_item = 'U'
            AND (desc_valor_novo IS NULL OR trim(desc_valor_novo) = ''))
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
UNION ALL
SELECT 'U (modificacao) com valor novo IGUAL ao anterior',
       COUNT_IF(tp_modificacao_item = 'U'
            AND trim(COALESCE(desc_valor_novo, '')) = trim(COALESCE(desc_valor_anterior, '')))
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
UNION ALL
SELECT 'ambos os valores vazios',
       COUNT_IF((desc_valor_novo IS NULL OR trim(desc_valor_novo) = '')
            AND (desc_valor_anterior IS NULL OR trim(desc_valor_anterior) = ''))
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
ORDER BY linhas DESC;

### 18.5 Indicador de alteração de valor — lógica invertida

O comentário declara: *"lógica invertida: X/1 => Não, demais => Sim"*.

Ou seja, o campo de origem `CDPOS-TEXT_CASE` foi **invertido** no pipeline.
Confira se o resultado bate com o esperado — e lembre que o extrato do SAP virá
com a lógica **original**, não invertida.

In [0]:
-- 18.5 INDICADOR DE ALTERACAO DE VALOR
SELECT COALESCE(NULLIF(trim(ind_alteracao_valor), ''), '(vazio)') AS ind_alteracao_valor,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
GROUP BY COALESCE(NULLIF(trim(ind_alteracao_valor), ''), '(vazio)')
ORDER BY linhas DESC;

### 18.6 Campos mais modificados

Quais campos da requisição mais mudam. Verifica também se o de-para
`nm_campo_modificado` → `desc_campo_modificado` é consistente — o comentário
declara que é um **mapa inline**, não um join com o dicionário SAP,
então campos novos podem ficar sem descrição.

In [0]:
-- 18.6 CAMPOS MAIS MODIFICADOS
SELECT nm_campo_modificado,
       desc_campo_modificado,
       COUNT(*)                                AS modificacoes,
       COUNT(DISTINCT num_requisicao_compra)   AS requisicoes_afetadas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
GROUP BY nm_campo_modificado, desc_campo_modificado
ORDER BY modificacoes DESC
LIMIT 40;

In [0]:
-- 18.6b CAMPOS SEM DESCRICAO NO MAPA INLINE
SELECT nm_campo_modificado,
       COUNT(DISTINCT desc_campo_modificado) AS qtd_descricoes,
       CONCAT_WS(' | ', SORT_ARRAY(COLLECT_SET(desc_campo_modificado))) AS descricoes,
       COUNT(*) AS linhas,
       CASE WHEN COUNT(DISTINCT desc_campo_modificado) > 1
            THEN 'ERRO - campo com mais de uma descricao'
            WHEN MAX(COALESCE(trim(desc_campo_modificado), '')) = ''
              OR MAX(desc_campo_modificado) = nm_campo_modificado
            THEN 'SEM DESCRICAO - fora do mapa inline'
            ELSE 'ok' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
GROUP BY nm_campo_modificado
ORDER BY qtd_descricoes DESC, linhas DESC
LIMIT 40;

### 18.7 Escopo declarado — classe de objeto e tabela

O comentário diz que `cod_classe_objeto` é **filtrado por BANF** e que `nm_tabela`
mapeia `EBAN` para *"Requisição de Compras"*.

Se ambos forem constantes, confirma que a tabela cobre **apenas** requisições —
limitação de escopo a registrar no relatório.

In [0]:
-- 18.7 ESCOPO: CLASSE DE OBJETO E TABELA
SELECT cod_classe_objeto, nm_tabela, desc_tabela,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
GROUP BY cod_classe_objeto, nm_tabela, desc_tabela
ORDER BY linhas DESC;

### 18.8 Dias entre solicitação e modificação

O comentário declara `abs(datediff(...))` — o resultado **não pode ser negativo**.
Valores muito altos merecem verificação.

In [0]:
-- 18.8 DIAS ENTRE SOLICITACAO E MODIFICACAO
SELECT COUNT(*)                                                       AS linhas,
       COUNT_IF(qt_dias_entre_solicitacao_modificacao < 0)            AS negativos_NAO_DEVERIA_EXISTIR,
       COUNT_IF(qt_dias_entre_solicitacao_modificacao = 0)            AS mesmo_dia,
       COUNT_IF(qt_dias_entre_solicitacao_modificacao IS NULL)        AS nulos,
       MIN(qt_dias_entre_solicitacao_modificacao)                     AS minimo,
       MAX(qt_dias_entre_solicitacao_modificacao)                     AS maximo,
       ROUND(AVG(qt_dias_entre_solicitacao_modificacao), 2)           AS media,
       percentile_approx(qt_dias_entre_solicitacao_modificacao, 0.5)  AS mediana,
       percentile_approx(qt_dias_entre_solicitacao_modificacao, 0.95) AS p95
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014';

### 18.9 Modificações por requisição e por usuário

Perfil da trilha de auditoria: quantas mudanças cada requisição sofreu e quem mais alterou.

In [0]:
-- 18.9 MODIFICACOES POR REQUISICAO
SELECT modificacoes_na_requisicao, COUNT(*) AS requisicoes
FROM (SELECT num_requisicao_compra, COUNT(*) AS modificacoes_na_requisicao
        FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014' GROUP BY num_requisicao_compra)
GROUP BY modificacoes_na_requisicao
ORDER BY modificacoes_na_requisicao
LIMIT 40;

In [0]:
-- 18.9b TOP USUARIOS MODIFICADORES
SELECT nm_usuario_modificador,
       COUNT(*)                                AS modificacoes,
       COUNT(DISTINCT num_requisicao_compra)   AS requisicoes,
       COUNT(DISTINCT num_documento)           AS documentos,
       MIN(dt_criacao_documento)               AS primeira_modificacao,
       MAX(dt_criacao_documento)               AS ultima_modificacao
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
GROUP BY nm_usuario_modificador
ORDER BY modificacoes DESC
LIMIT 30;

### 18.10 Documento de modificação — campos por documento

Um documento `CHANGENR` agrupa todos os campos alterados numa mesma transação.
Confirma a granularidade real da tabela.

In [0]:
-- 18.10 CAMPOS POR DOCUMENTO DE MODIFICACAO
SELECT campos_no_documento, COUNT(*) AS documentos
FROM (SELECT num_documento, COUNT(*) AS campos_no_documento
        FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014' GROUP BY num_documento)
GROUP BY campos_no_documento
ORDER BY campos_no_documento
LIMIT 40;

In [0]:
-- 18.10b EXEMPLO: UM DOCUMENTO COM VARIOS CAMPOS
WITH alvo AS (
  SELECT num_documento
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
  GROUP BY num_documento
  ORDER BY COUNT(*) DESC
  LIMIT 1
)
SELECT d.num_documento, d.num_requisicao_compra, d.num_item_requisicao_compra,
       d.dt_criacao_documento, d.hr_modificacao, d.nm_usuario_modificador,
       d.nm_campo_modificado, d.desc_campo_modificado,
       d.desc_valor_anterior, d.desc_valor_novo,
       d.tp_modificacao_item, d.desc_tipo_modificacao_item
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra d
JOIN alvo a ON d.num_documento = a.num_documento
ORDER BY d.nm_campo_modificado;

## 19. EXTRAÇÃO COMPLETA — centro 4014

**Esta é a célula que você baixa para comparar com o SAP.**

Após executar, use **Download → CSV** no resultado.

In [0]:
-- 19. EXTRACAO COMPLETA DO CENARIO
SELECT *
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'
ORDER BY `num_requisicao_compra`, `num_item_requisicao_compra`, `num_documento`;

### 19.1 Alternativa para volume grande _(opcional)_

In [0]:
-- 19.1 GRAVAR EXTRACAO EM TABELA (opcional)
-- CREATE OR REPLACE TABLE dev_procurement.corp_curated.extracao_zglmm379_4014 AS
-- SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra WHERE `cod_centro` = '4014';
--
-- SELECT COUNT(*) FROM dev_procurement.corp_curated.extracao_zglmm379_4014;
SELECT 'Descomente as linhas acima se precisar gravar a extracao em tabela' AS instrucao;

## 20. Resumo

**Copie esta saída** e envie ao agente junto com o notebook.

In [0]:
-- 20. RESUMO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014'),
g AS (
  SELECT 'num_documento' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_documento` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_requisicao_compra + num_documento' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_requisicao_compra`, `num_documento` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_documento + nm_campo_modificado' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_documento`, `nm_campo_modificado` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_requisicao_compra + num_item_requisicao_compra + num_documento' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_requisicao_compra`, `num_item_requisicao_compra`, `num_documento` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014')
UNION ALL
  SELECT 'num_requisicao_compra + num_item_requisicao_compra + num_documento + nm_campo_modificado' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_requisicao_compra`, `num_item_requisicao_compra`, `num_documento`, `nm_campo_modificado` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm379_gestao_requisicao_compra
WHERE `cod_centro` = '4014')
)
SELECT 'CENARIO' AS bloco, 'transacao' AS item, 'ZGLMM379' AS valor
UNION ALL SELECT 'CENARIO', 'tabela', 'tbl_ds_pro_zglmm379_gestao_requisicao_compra'
UNION ALL SELECT 'CENARIO', 'filtro', 'cod_centro = 4014'
UNION ALL SELECT 'CENARIO', 'linhas', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'CENARIO', 'colunas', '25'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Baixar a **seção 19** em CSV — é a base do centro 4014 no Datalake.
2. Extrair a ZGLMM379 no SAP com o filtro `centro = 4014`.
3. Anotar a data e hora das duas extrações.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Antes de comparar — atenção redobrada nesta tabela

- [ ] **Data em `dd/MM/yyyy`** — converter o lado SAP antes de comparar (seção 18.2)
- [ ] **Requisição e item já sem zeros à esquerda** — normalizar o lado **SAP** (seção 11)
- [ ] **`ind_alteracao_valor` invertido** — o SAP traz a lógica original (seção 18.5)
- [ ] Chave real confirmada — inclui `nm_campo_modificado`? (seção 4)
- [ ] Projeção de `cod_chave_tabela` validada (seção 18.1)
- [ ] Colunas 100% nulas conferidas no SAP (seção 6)
